In [5]:
import os
import sys

from pathlib import Path
from tempfile import TemporaryDirectory

import pymupdf
from tqdm.auto import tqdm
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling_core.types.doc import ImageRefMode

In [ ]:
def pages_to_markdown(
    pdf_path,
    page_start=None,
    page_end=None,
    output_dir=Path('/Users/Advay/Desktop/Relativity/users/advay/qg/parsed'),
):
    pdf_path = Path(pdf_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pp = PdfPipelineOptions(do_table_structure=True)
    pp.table_structure_options.mode = TableFormerMode.ACCURATE
    pp.table_structure_options.do_cell_matching = True
    pp.images_scale = 4.0
    pp.generate_page_images = True
    pp.do_formula_enrichment = True

    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pp)}
    )

    with pymupdf.open(pdf_path) as source_doc, TemporaryDirectory() as tmp_dir:
        tmp_dir = Path(tmp_dir)
        total_pages = len(source_doc)

        start = 0 if page_start is None else page_start
        end = total_pages if page_end is None else page_end

        if not 0 <= start <= total_pages:
            raise ValueError(f'page_start must be between 0 and {total_pages}')
        if not 0 <= end <= total_pages:
            raise ValueError(f'page_end must be between 0 and {total_pages}')
        if start >= end:
            raise ValueError('page_start must be smaller than page_end')

        for page_idx in tqdm(range(start, end), desc='Converting pages'):

            output_path = output_dir / f'page_{page_idx + 1:04d}.md'
            if os.path.exists(output_path):
                continue

            single_page_pdf = tmp_dir / f'page_{page_idx + 1:04d}.pdf'

            with pymupdf.open() as single_page_doc:
                single_page_doc.insert_pdf(source_doc, from_page=page_idx, to_page=page_idx)
                single_page_doc.save(single_page_pdf)

            res = converter.convert(str(single_page_pdf), raises_on_error=False)
            md = res.document.export_to_markdown(image_mode=ImageRefMode.EMBEDDED)

            output_path.write_text(md, encoding='utf-8')


In [1]:
# pages_to_markdown("QG.pdf")